# ARBench on Google Colab (L4 GPU)

Benchmarks **Granite 4.1, Gemma 4 and Qwen 3** (via Ollama) on your annual report:
YourBench-style benchmark generation → LightEval-style evaluation → per-model + comparison reports.

**Before you start:** `Runtime → Change runtime type → L4 GPU`.

Steps: 1) get the project · 2) install Ollama · 3) add the annual report · 4) run (one command) · 5) download results.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Get the project

**Option A (simplest):** zip the project folder on your laptop (without `.venv`), drag `ARBench.zip` into Colab's
**Files** pane (left sidebar → upload). If no zip is there, the cell asks you to upload one.
**Option B:** upload the project folder to Google Drive (e.g. `MyDrive/ARBench`) and set `USE_DRIVE = True`.

In [ ]:
import os, zipfile, glob
USE_DRIVE = False                                  # True -> use the project folder on Google Drive
PROJECT_DIR = "/content/drive/Othercomputers/My laptop/Lectures/Year 2/DSGP/Benchmarking SLMs/Local Benchmarking with Evals"      # (Drive only) folder that contains run_benchmark.py

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    parts = sorted(glob.glob("/content/ARBench.zip.part*"))
    if parts and not os.path.exists("/content/ARBench.zip"):       # browser uploads are size-limited -> split zip
        with open("/content/ARBench.zip", "wb") as out:
            for part in parts:
                out.write(open(part, "rb").read())
        print("Joined", len(parts), "parts")
    zips = sorted(glob.glob("/content/ARBench*.zip") + glob.glob("/content/*.zip"))
    if not zips:
        from google.colab import files
        print("Upload ARBench.zip (the zipped project folder):")
        zips = ["/content/" + next(iter(files.upload()))]
    print("Extracting", zips[0])
    zipfile.ZipFile(zips[0]).extractall("/content/ARBench_src")
    hits = glob.glob("/content/ARBench_src/**/run_benchmark.py", recursive=True)
    assert hits, "run_benchmark.py not found inside the zip"
    PROJECT_DIR = os.path.dirname(hits[0])

assert os.path.exists(os.path.join(PROJECT_DIR, "run_benchmark.py")), f"run_benchmark.py not found in {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print("Project:", os.getcwd())

## 2 · Install Ollama + Python dependencies and start the server (GPU)

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install -r requirements.txt

In [ ]:
import subprocess, time, requests, os
env = dict(os.environ,
           OLLAMA_NUM_PARALLEL="4",        # parallel requests per model (matches --profile colab)
           OLLAMA_MAX_LOADED_MODELS="3",   # all three SLMs fit in 24 GB VRAM
           OLLAMA_KEEP_ALIVE="30m",
           OLLAMA_FLASH_ATTENTION="1")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("Ollama", requests.get("http://127.0.0.1:11434/api/version", timeout=2).json()); break
    except Exception:
        time.sleep(1)

## 3 · Choose the models and add the annual report

The defaults are the same small models as the laptop run. On the L4 you can also try larger sizes of the same families.

In [ ]:
MODELS = ["granite4.1:3b", "gemma4:e2b-it-qat", "qwen3:4b-instruct"]
# Larger variants that fit an L4 comfortably:
# MODELS = ["granite4.1:8b", "gemma4:e4b-it-qat", "qwen3:8b"]
import subprocess
for m in MODELS:
    for attempt in range(5):                       # retry flaky downloads
        if subprocess.run(["ollama", "pull", m]).returncode == 0:
            break
    else:
        raise RuntimeError(f"could not pull {m}")
!ollama list

In [ ]:
import os, shutil
REPORT_DIR = "data/annual_report"
os.makedirs(REPORT_DIR, exist_ok=True)
existing = [f for f in os.listdir(REPORT_DIR) if f.lower().endswith((".pdf", ".docx", ".html", ".htm", ".md", ".txt")) and f.lower() != "readme.md"]
if not existing:
    from google.colab import files
    print("Upload the annual report (PDF/DOCX/HTML/MD/TXT):")
    for name, data in files.upload().items():
        open(os.path.join(REPORT_DIR, name), "wb").write(data)
print("Reports:", [f for f in os.listdir(REPORT_DIR) if f.lower() != "readme.md"])

## 4 · Run the whole benchmark (one command)

`--profile colab` = full coverage (≈60 single-hop chunks, multi-hop, MCQ) with 4 parallel requests.
Use `--profile quick` for a fast first look. Everything is cached, so re-running the cell resumes.

In [ ]:
!python run_benchmark.py --profile colab --run-name colab_run --models {" ".join(MODELS)}

## 5 · Results

In [ ]:
from IPython.display import Markdown, display, HTML
run = "outputs/colab_run"
display(Markdown(open(f"{run}/reports/comparison_report.md", encoding="utf-8").read().replace("](charts/", f"]({run}/reports/charts/")))

In [ ]:
from IPython.display import Image
for c in ["leaderboard", "tasks", "quality_vs_speed", "hallucination", "question_types", "difficulty"]:
    p = f"outputs/colab_run/reports/charts/{c}.png"
    if os.path.exists(p): display(Image(p))

In [ ]:
import shutil
shutil.make_archive("/content/arbench_results", "zip", "outputs/colab_run")
from google.colab import files
files.download("/content/arbench_results.zip")   # also saved inside PROJECT_DIR/outputs if you used Drive